# Offline Prep (Model Weights Only)

**Scope**: download HF model weights into `/kaggle/working/` for snapshotting.
The codebase (cloned `main`) is fetched into `/tmp/` so it does
**not** bundle into this notebook's output Dataset.

**Why this split**: the codebase has its own prep notebook
(`prep_codebase.ipynb`) which produces a small (~few MB) code-only Dataset.
Splitting model weights from codebase means iterating on codebase changes
doesn't require re-downloading the model (~16 GB, hours-long step).

**Output Dataset of this notebook**: just `Meta-Llama-3-8B-Instruct/` (or
the equivalent for `aya` / `qwen`). Attach this Dataset + the prep-codebase
Dataset to consumer notebooks.

**Pre-requisites** (Kaggle Secrets):
- `GITHUB_TOKEN` — to clone the private/public repo
- `HF_TOKEN` — to download gated model weights


## 1. Configuration

In [ ]:
# ── Pick ONE: "llama", "aya", or "qwen" ──
MODEL_TARGET = "llama"

# Repo to clone for the prep module + (optionally) for inclusion in the snapshot.
REPO_HTTPS = "github.com/ANONYMOUS/ANONYMIZED-REPO.git"
REPO_BRANCH = "main"  # branch that contains prep/download_model.py

## 2. Auth — Kaggle Secrets + HF login

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN     = secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)
print("Secrets loaded; HF login OK.")

## 3. Clone the repo to get the `prep` module

In [ ]:
# Where the model weights land (becomes the output Dataset)
WORKING = Path("/kaggle/working")

# Clone the repo to /tmp (NOT /kaggle/working) so the codebase does NOT bundle
# into this notebook's output Dataset. The codebase has its own prep notebook
# (prep_codebase.ipynb) which produces a separate, smaller, code-only Dataset.
# This notebook's output is just the model weights (~16 GB), nothing else.

REPO_DIR = Path("/tmp/layerselection-repo")    # ← /tmp/, NOT /kaggle/working/

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
     f"https://{GITHUB_TOKEN}@{REPO_HTTPS}", str(REPO_DIR)],
    check=True,
)

assert (REPO_DIR / "prep" / "download_model.py").exists(), (
    f"prep/download_model.py not found at the root of {REPO_DIR} "
    f"— check REPO_BRANCH (currently {REPO_BRANCH!r})"
)
sys.path.insert(0, str(REPO_DIR))
print(f"prep module loaded from: {REPO_DIR} (excluded from output Dataset)")
print(f"WORKING (output dir): {WORKING}")


## 4. Download model weights (~16 GB)

In [ ]:
from prep.download_model import download_model, report_snapshot_size, resolve_model_spec

spec = resolve_model_spec(MODEL_TARGET)
print(f"MODEL_TARGET: {MODEL_TARGET}")
print(f"HF repo:      {spec['hf_repo_id']}")
print(f"Local dir:    {WORKING / spec['local_dir_name']}")
print()

model_dir = download_model(MODEL_TARGET, WORKING, HF_TOKEN)
print(f"\nFiles in {model_dir}:")
report_snapshot_size(model_dir)

## 5. Verify snapshot

In [ ]:
from prep.download_model import verify_model_snapshot

ok, missing = verify_model_snapshot(model_dir)
if ok:
    # Only count files inside /kaggle/working/ (excludes /tmp/layerselection-repo)
    working = Path("/kaggle/working")
    total_gb = sum(f.stat().st_size for f in working.rglob("*") if f.is_file()) / 1e9
    print(f"OK — snapshot ready. /kaggle/working/ total: {total_gb:.2f} GB")
    print(f"  (this should be ~weights size only — the cloned repo is in /tmp/, excluded from output)")
    print(f"\nNext: Save Version → Save & Run All (Commit).")
    print(f"Then change MODEL_TARGET in cell 1 and commit a second version.")
else:
    raise RuntimeError(f"Snapshot incomplete; missing globs: {missing}")
